# Robust Quadruped Training

This notebook implements robust training with **atomic curriculum callbacks**:
- **Terrain Curriculum** - Progressively harder terrain (TerrainCurriculumCallbackV2)
- **Force Curriculum** - Increasing push perturbations (ForceCurriculumCallback)
- **Friction Curriculum** - Widening friction range (FrictionCurriculumCallback)
- **Motor Noise Curriculum** - Increasing actuator noise (MotorNoiseCurriculumCallback)
- **Sensor Noise Curriculum** - Increasing observation noise (SensorNoiseCurriculumCallback)

All callbacks follow **atomic design principles** - each handles ONE aspect of curriculum learning.
All are `@configurable` for easy parameter saving/loading.

Based on best practices from ETH RSL's legged_gym framework.

In [3]:
import sys
import time
from pathlib import Path
sys.path.insert(0, "../../..")

import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.logger import configure

from spotmicro.env.spotmicro_env import SpotmicroEnv
from spotmicro.physics.factory import create_backend
from spotmicro.devices.random_controller import RandomController
from spotmicro.tools.config import Config

# Atomic curriculum callbacks (all @configurable)
from training.callbacks import (
    TerrainCurriculumCallbackV2,
    ForceCurriculumCallback,
    FrictionCurriculumCallback,
    MotorNoiseCurriculumCallback,
    SensorNoiseCurriculumCallback,
)

print("Imports successful!")

Imports successful!


In [4]:
# Import reward function
from reward_function import reward_function, RewardState, RewardConfig

print("Reward function loaded!")

Reward function loaded!


## Configuration

All parameters are managed through the central Config registry.
Use `obj.save('config.yaml')` to save and `obj.load('config.yaml')` to restore.

In [5]:
# Training parameters
TOTAL_STEPS = 100_000
RUN_NAME = "robust_walk_v1"
LOG_DIR = f"./logs/{RUN_NAME}"

# Central config registry
cfg = Config()

# Reward configuration (legged_gym defaults)
reward_config = RewardConfig(
    tracking_lin_vel=1.0,
    tracking_ang_vel=0.5,
    tracking_sigma=0.25,
    feet_air_time=1.0,
    lin_vel_z=-2.0,
    ang_vel_xy=-0.05,
    orientation=-1.0,
    action_rate=-0.01,
    torques=-0.00001,
)

print("Configuration loaded!")
print(f"Run name: {RUN_NAME}")
print(f"Total steps: {TOTAL_STEPS:,}")

Configuration loaded!
Run name: robust_walk_v1
Total steps: 100,000


## Environment Setup

In [6]:
# Create environment
device = RandomController(cfg)  # Random velocity commands
backend = create_backend("pybullet", use_gui=False)

env = SpotmicroEnv(
    backend=backend,
    device=device,
    config=cfg,
    reward_fn=reward_function,
    reward_state=RewardState(reward_config),
    use_gui=False,
    max_episode_len=3000,
)

# Verify environment
check_env(env, warn=True)
print(f"Environment created! Obs space: {env.observation_space.shape}, Action space: {env.action_space.shape}")

Initializing base state
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
front_left_leg_link_coverb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
front_right_leg_link_coverb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
rear_left_leg_link_coverb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local iner

## Atomic Callbacks Setup

Each callback handles ONE aspect of curriculum learning.
All are `@configurable` - parameters can be saved/loaded from YAML.

In [7]:
class WallClockCheckpointCallback(BaseCallback):
    """Save periodic checkpoints based on elapsed wall-clock time."""

    def __init__(self, save_path, run_name, config, interval_seconds=300, verbose=1):
        super().__init__(verbose)
        self.save_path = Path(save_path)
        self.run_name = run_name
        self.config = config
        self.interval_seconds = interval_seconds
        self._start_time = None
        self._last_save_time = None
        self._save_index = 0

    def _on_training_start(self):
        self.save_path.mkdir(parents=True, exist_ok=True)
        now = time.time()
        self._start_time = now
        self._last_save_time = now

    def _on_step(self):
        now = time.time()
        if now - self._last_save_time < self.interval_seconds:
            return True

        self._save_index += 1
        elapsed_minutes = int((now - self._start_time) // 60)
        save_file = self.save_path / (
            f"ppo_{self.run_name}_timed_{self._save_index:03d}_{elapsed_minutes:04d}min"
        )
        snapshot_file = save_file.with_suffix(".yaml")
        self.model.save(save_file)
        self.config.save(str(snapshot_file))
        self._last_save_time = now

        if self.verbose:
            print(f"[TimedCheckpoint] Saved model to: {save_file}.zip")
            print(f"[TimedCheckpoint] Saved snapshot to: {snapshot_file}")
        return True

# Checkpoint callback - save every 5% of training
checkpoint_callback = CheckpointCallback(
    save_freq=TOTAL_STEPS // 20,
    save_path=f"{RUN_NAME}_checkpoints",
    name_prefix=f"ppo_{RUN_NAME}"
)

# Timed checkpoint callback - save a testable policy every 5 real minutes
timed_checkpoint_callback = WallClockCheckpointCallback(
    save_path=f"{RUN_NAME}_timed_checkpoints",
    run_name=RUN_NAME,
    config=cfg,
    interval_seconds=300,
    verbose=1,
)

# === Atomic Curriculum Callbacks ===

# 1. Terrain Curriculum - starts flat, increases height variation
terrain_callback = TerrainCurriculumCallbackV2(
    config=cfg,
    env=env,
    total_timesteps=TOTAL_STEPS,
    z_max_initial=0.02,      # Nearly flat
    z_max_final=0.3,         # Full height variation
    change_every_episodes=50,
    schedule="linear",
    warmup_ratio=0.05,
    verbose=False,
)

# 2. Force Curriculum - starts gentle, increases push strength
force_callback = ForceCurriculumCallback(
    config=cfg,
    env=env,
    total_timesteps=TOTAL_STEPS,
    push_vel_initial=0.1,    # Gentle pushes
    push_vel_final=1.5,      # Strong pushes
    push_interval_s=15.0,
    schedule="linear",
    warmup_ratio=0.05,
    verbose=False,
)

# 3. Friction Curriculum - starts narrow, widens range
friction_callback = FrictionCurriculumCallback(
    config=cfg,
    env=env,
    total_timesteps=TOTAL_STEPS,
    friction_initial_low=0.9,
    friction_initial_high=1.1,
    friction_final_low=0.4,
    friction_final_high=1.5,
    schedule="linear",
    warmup_ratio=0.05,
    verbose=False,
)

# 4. Motor Noise Curriculum - starts perfect, adds actuator noise
motor_noise_callback = MotorNoiseCurriculumCallback(
    config=cfg,
    env=env,
    total_timesteps=TOTAL_STEPS,
    noise_initial=0.0,       # Perfect actuators
    noise_final=0.05,        # Noisy actuators (rad)
    noise_type="gaussian",
    schedule="linear",
    warmup_ratio=0.05,
    verbose=False,
)

# 5. Sensor Noise Curriculum - starts clean, adds observation noise
sensor_noise_callback = SensorNoiseCurriculumCallback(
    config=cfg,
    env=env,
    total_timesteps=TOTAL_STEPS,
    noise_scale_initial=0.0, # Clean observations
    noise_scale_final=1.0,   # Full noise (legged_gym defaults)
    dof_pos_noise=0.01,      # Base noise levels
    dof_vel_noise=1.5,
    lin_vel_noise=0.1,
    ang_vel_noise=0.2,
    schedule="linear",
    warmup_ratio=0.05,
    verbose=False,
)

# Combine all callbacks
callbacks = CallbackList([
    checkpoint_callback,
    timed_checkpoint_callback,
    terrain_callback,
    force_callback,
    friction_callback,
    motor_noise_callback,
    sensor_noise_callback,
])

print("Atomic callbacks configured!")
print(f"  - TerrainCurriculumCallbackV2: z_max {terrain_callback.z_max_initial} -> {terrain_callback.z_max_final}")
print(f"  - ForceCurriculumCallback: push_vel {force_callback.push_vel_initial} -> {force_callback.push_vel_final}")
print(f"  - FrictionCurriculumCallback: range {friction_callback.friction_initial} -> {friction_callback.friction_final}")
print(f"  - MotorNoiseCurriculumCallback: noise {motor_noise_callback.noise_initial} -> {motor_noise_callback.noise_final}")
print(f"  - SensorNoiseCurriculumCallback: scale {sensor_noise_callback.noise_scale_initial} -> {sensor_noise_callback.noise_scale_final}")

Atomic callbacks configured!
  - TerrainCurriculumCallbackV2: z_max 0.02 -> 0.3
  - ForceCurriculumCallback: push_vel 0.1 -> 1.5
  - FrictionCurriculumCallback: range (0.9, 1.1) -> (0.4, 1.5)
  - MotorNoiseCurriculumCallback: noise 0.0 -> 0.05
  - SensorNoiseCurriculumCallback: scale 0.0 -> 1.0


## Save Configuration

Save all configurable callback parameters to YAML for reproducibility.

In [8]:
# Save the current configurable snapshot to YAML
config_path = f"{RUN_NAME}_config.yaml"

# The shared Config registry already holds every configurable component
cfg.save(config_path)

print(f"Configuration saved to: {config_path}")

Configuration saved to: robust_walk_v1_config.yaml


## Model Setup

In [9]:
def clipped_linear_schedule(initial_value, min_value=1e-5):
    """Linear learning rate schedule with minimum clip."""
    def schedule(progress_remaining):
        return max(progress_remaining * initial_value, min_value)
    return schedule

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=clipped_linear_schedule(3e-4),
    ent_coef=0.01,      # Slightly higher for exploration
    clip_range=0.2,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    tensorboard_log=LOG_DIR,
    device='cpu'
)

# Configure logger
new_logger = configure(LOG_DIR, ["stdout", "csv", "tensorboard"])
model.set_logger(new_logger)

print(f"Model created! Policy: {model.policy}")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/robust_walk_v1
Model created! Policy: ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=97, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=97, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=12, bias=True)
  (value_net): Linear(in_features=64, out_featur

## Training

In [8]:
print(f"Starting training for {TOTAL_STEPS:,} steps...")
print(f"Logging to: {LOG_DIR}")
print(f"Checkpoints every {TOTAL_STEPS // 20:,} steps")
print(f"Timed checkpoints every 5 real minutes: ./{RUN_NAME}_timed_checkpoints")

try:
    model.learn(
        total_timesteps=TOTAL_STEPS,
        callback=callbacks,
        log_interval=1,
        reset_num_timesteps=True,
    )
except KeyboardInterrupt:
    interrupted_path = f"ppo_{RUN_NAME}_interrupted"
    interrupted_snapshot = f"{interrupted_path}.yaml"
    model.save(interrupted_path)
    cfg.save(interrupted_snapshot)
    print(f"\nTraining interrupted. Emergency save written to: {interrupted_path}.zip")
    print(f"Training snapshot written to: {interrupted_snapshot}")
    raise

# Save final model
final_model_path = f"ppo_{RUN_NAME}"
final_snapshot_path = f"{final_model_path}.yaml"
model.save(final_model_path)
cfg.save(final_snapshot_path)
print(f"\nTraining complete! Model saved to: {final_model_path}.zip")
print(f"Training snapshot saved to: {final_snapshot_path}")

Starting training for 100,000 steps...
Logging to: ./logs/robust_walk_v1
Checkpoints every 5,000 steps
Timed checkpoints every 5 real minutes: ./robust_walk_v1_timed_checkpoints
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 20.8     |
|    ep_rew_mean     | -210     |
| time/              |          |
|    fps             | 557      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 20.2        |
|    ep_rew_mean          | -209        |
| time/                   |             |
|    fps                  | 448         |
|    iterations           | 2           |
|    time_elapsed         | 9           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011230422 |
|    clip_fraction  

In [9]:
# Cleanup
env.close()
print("Environment closed.")

Environment closed.


## Evaluation

In [ ]:
# Choose a saved model and matching snapshot YAML
SNAPSHOT_MODEL_PATH = f"ppo_{RUN_NAME}"
SNAPSHOT_CONFIG_PATH = f"{SNAPSHOT_MODEL_PATH}.yaml"

# Create evaluation environment with GUI
eval_cfg = Config(SNAPSHOT_CONFIG_PATH)
eval_backend = create_backend("pybullet", use_gui=True)
eval_device = RandomController(eval_cfg)

eval_env = SpotmicroEnv(
    backend=eval_backend,
    device=eval_device,
    config=eval_cfg,
    reward_fn=reward_function,
    reward_state=RewardState(reward_config),
    use_gui=True,
    max_episode_len=3000,
)

# Restore the saved curriculum snapshot in the evaluation environment
terrain_eval_callback = TerrainCurriculumCallbackV2(config=eval_cfg, env=eval_env, verbose=False)
force_eval_callback = ForceCurriculumCallback(config=eval_cfg, env=eval_env, verbose=False)
friction_eval_callback = FrictionCurriculumCallback(config=eval_cfg, env=eval_env, verbose=False)
motor_noise_eval_callback = MotorNoiseCurriculumCallback(config=eval_cfg, env=eval_env, verbose=False)
sensor_noise_eval_callback = SensorNoiseCurriculumCallback(config=eval_cfg, env=eval_env, verbose=False)

terrain_eval_callback.apply_saved_state(eval_env)
friction_eval_callback.apply_saved_state(eval_env)
motor_noise_eval_callback.apply_saved_state()
sensor_noise_eval_callback.apply_saved_state()

# Load trained model
model = PPO.load(SNAPSHOT_MODEL_PATH, env=eval_env)
print(f"Model loaded for evaluation from: {SNAPSHOT_MODEL_PATH}.zip")
print(f"Snapshot restored from: {SNAPSHOT_CONFIG_PATH}")

In [ ]:
# Run evaluation episodes
n_eval_episodes = 5
episode_rewards = []

for ep in range(n_eval_episodes):
    obs, info = eval_env.reset()
    ep_reward = 0
    done = False
    
    while not done:
        force_eval_callback.step_saved_state(eval_env)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += reward
        done = terminated or truncated
    
    episode_rewards.append(ep_reward)
    print(f"Episode {ep + 1}: Reward = {ep_reward:.2f}")

print(f"\nMean reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")

Episode 1: Reward = -31.14
Episode 2: Reward = -14.68
Episode 3: Reward = -6.87
Episode 4: Reward = -104.55
Episode 5: Reward = -105.33

Mean reward: -52.51 +/- 43.52


X connection to :0 broken (explicit kill or server shutdown).


: 

In [ ]:
# Cleanup
eval_env.close()
print("Evaluation complete!")